In [10]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("/kaggle/input/datasets/zvozvozvo/movielenseda/ratings.csv")

df.rating = df.rating.astype(int)

df.head(5)

,userId,movieId,rating,timestamp
0,1,17,4,944249077
1,1,25,1,944250228
2,1,29,2,943230976
3,1,30,5,944249077
4,1,32,5,943228858


In [11]:
# Разделим на трейн и тест по 01.01.2018

train_df = df.loc[(df.timestamp < 1514764800)].copy()
test_df = df.loc[(df.timestamp >= 1514764800)].copy()

In [12]:
train_users = train_df.userId.unique()
test_users = test_df.userId.unique()

# Возьмем только тех людей, которые попали и в трейн, и в тест
all_included = np.intersect1d(train_users, test_users)

n_users = all_included.shape[0]

# В трейне и тесте выделим только тех, кто есть и там, и там
train_df = train_df.loc[train_df.userId.isin(all_included)].copy()
test_df = test_df.loc[test_df.userId.isin(all_included)].copy()

# Сгруппируем все интеракции по пользователям
train_grouped = train_df.groupby('userId').apply(
    lambda x: [(t1, t2) for t1, t2 in sorted(zip(x.movieId, 
                                                 x.rating), key=lambda x: x[1])]
).reset_index()
train_grouped.rename({0:'train_interactions'}, axis=1, inplace=True)

test_grouped = test_df.groupby('userId').apply(
    lambda x: [(t1, t2) for t1, t2 in sorted(zip(x.movieId,
                                                         x.rating), key=lambda x: x[1])]
).reset_index()
test_grouped.rename({0:'test_interactions'}, axis=1, inplace=True)

train_grouped.head()

/tmp/ipykernel_55/506008888.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_grouped = train_df.groupby('userId').apply(
/tmp/ipykernel_55/506008888.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_grouped = test_df.groupby('userId').apply(


,userId,train_interactions
0,28,"[(8666, 0), (54, 1), (75, 1), (160, 1), (174, ..."
1,35,"[(1247, 2), (1590, 2), (1917, 2), (6365, 2), (..."
2,100,"[(49274, 1), (60069, 1), (34, 2), (1127, 2), (..."
3,111,"[(104, 2), (288, 2), (356, 2), (2424, 2), (316..."
4,140,"[(94959, 2), (168254, 2), (6870, 3), (56587, 3..."


In [13]:
joined = train_grouped.merge(test_grouped)
joined

,userId,train_interactions,test_interactions
0,28,"[(8666, 0), (54, 1), (75, 1), (160, 1), (174, ...","[(4125, 0), (4138, 0), (261759, 1), (5679, 2),..."
1,35,"[(1247, 2), (1590, 2), (1917, 2), (6365, 2), (...","[(2513, 2), (5218, 2), (5618, 2), (7115, 2), (..."
2,100,"[(49274, 1), (60069, 1), (34, 2), (1127, 2), (...","[(8533, 1), (222477, 1), (608, 2), (1183, 2), ..."
3,111,"[(104, 2), (288, 2), (356, 2), (2424, 2), (316...","[(168330, 3), (158238, 4)]"
4,140,"[(94959, 2), (168254, 2), (6870, 3), (56587, 3...","[(160954, 3), (176101, 5), (177593, 5), (17776..."
...,...,...,...
8562,200898,"[(4306, 3), (115713, 3), (122904, 3), (152081,...","[(2662, 2), (6534, 3), (78469, 3), (85414, 3),..."
8563,200915,"[(88267, 0), (18, 1), (592, 1), (1676, 1), (21...","[(60069, 5)]"
8564,200920,"[(159721, 0), (56949, 1), (63992, 1), (72407, ...","[(179819, 3), (122906, 4), (122912, 5)]"
8565,200930,"[(2706, 0), (4718, 0), (122904, 0), (171729, 0...","[(112556, 0), (140880, 0), (176149, 0), (92259..."


In [14]:
# Попробуем всем рекомендовать наиболее полпулярные айтемы
class TopPopular:

    def __init__(self):

        self.trained = False
    
    def fit(self, df, col='train_interactions'):
        # Получаем словарь всех фильмов с количеством их оценок
        counts = {}
        for _, row in df.iterrows():
            for item, _ in row[col]:
                if item in counts:
                    counts[item] += 1
                else:
                    counts[item] = 1
        return counts


    
toppop = TopPopular()
most_pop = toppop.fit(joined)

In [15]:
# Будем использовать популярность фильма как признак для бустинга

movies = pd.read_csv('/kaggle/input/datasets/zvozvozvo/movielenseda/movies.csv')
movies['genres'] = movies.genres.str.split("|")

movies["popularity"] = movies.movieId.apply(lambda x: most_pop.get(x, 0))

# Отнормируем
total = sum(movies['popularity'])
movies['popularity'] = movies['popularity'].apply(lambda x: x / total)

# Используем год выхода фильма
movies["age"] = 2025 - movies["title"].str.extract(r"\((\d{4})\)").astype(float).astype("Int64")

movies['title'] = movies.title.str[:-7]

# Добавим среднюю оценку фильма
movies = movies.merge((df[["rating", "movieId"]]
               .groupby(["movieId"]).mean().reset_index()), on='movieId')

In [16]:
links = pd.read_csv('/kaggle/input/datasets/zvozvozvo/movielenseda/links.csv')

# Добавим метаданные о фильмах
basic = pd.read_parquet("/kaggle/input/datasets/zvozvozvo/movielenseda/data_full.parquet")

basic["imdbId"] = basic["tconst"].str.replace("tt", "").astype(int)
basic = (basic.merge(links[["movieId", "imdbId"]], on="imdbId", how="right")
         .drop(columns=["tconst", "primaryTitle", "startYear", "genres", "imdbId", "decade", "averageRating", "numVotes", "ordering"]))

counts = basic.primaryName.value_counts().reset_index()

# Фильтруем строки, где количество больше 30
persons = counts[counts['count'] > 30].primaryName.unique().tolist()

basic = basic[basic.category.isin(['actor', 'actress', 'director']) & (basic.star_power == 4) & basic.primaryName.isin(persons)].drop(columns=['knownForTitles', 'nconst', 'has_original_title'])

In [17]:
def process_movie_group(group):
    # Собираем всех actor и actress в один список
    actors = group[group['category'].isin(['actor', 'actress'])]['primaryName'].unique().tolist()
    
    # Находим режиссера (берем первого найденного, если их несколько)
    director = group[group['category'] == 'director']['primaryName'].unique()
    director_name = director[0] if len(director) > 0 else None
    
    res = group.iloc[0].copy()
    
    # Обновляем нужные поля
    res['actors_list'] = actors
    res['director_name'] = director_name
    
    return res

# Группируем по movieId и применяем функцию
final_df = basic.groupby('movieId').apply(process_movie_group).reset_index(drop=True)

final_df = final_df.drop(columns=['category', 'primaryName'])

final_df = final_df.drop(columns=['star_power'])
final_df.is_multiregional = final_df.is_multiregional.astype(bool)

movies = movies.merge(final_df, on='movieId', how='left')

/tmp/ipykernel_55/4063384548.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df = basic.groupby('movieId').apply(process_movie_group).reset_index(drop=True)


In [18]:
movies

,movieId,title,genres,popularity,age,rating,runtimeMinutes,num_translations,main_region,is_multiregional,actors_list,director_name
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",0.001176,30,3.778556,81.0,66.0,IN,True,"[Tom Hanks, Tim Allen, Wallace Shawn, John Mor...",None
1,2,Jumanji,"[Adventure, Children, Fantasy]",0.000645,30,3.143925,104.0,54.0,CA,True,"[Robin Williams, Kirsten Dunst, David Alan Gri...",None
2,3,Grumpier Old Men,"[Comedy, Romance]",0.000120,30,3.070352,101.0,35.0,JP,True,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",None
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",0.000027,30,2.803635,124.0,20.0,US,True,"[Angela Bassett, Loretta Devine, Dennis Haysbe...",Forest Whitaker
4,5,Father of the Bride Part II,[Comedy],0.000155,30,2.984719,106.0,34.0,US,True,"[Steve Martin, Diane Keaton, Martin Short, Jan...",None
...,...,...,...,...,...,...,...,...,...,...,...,...
84427,292731,The Monroy Affaire,[Drama],0.000000,3,4.000000,95.0,3.0,MX,True,[Damián Alcázar],None
84428,292737,Shelter in Solitude,"[Comedy, Drama]",0.000000,2,1.000000,93.0,7.0,AU,True,[Robert Patrick],None
84429,292753,Orca,[Drama],0.000000,2,4.000000,NaN,NaN,NaN,NaN,NaN,NaN
84430,292755,The Angry Breed,[Drama],0.000000,57,1.000000,94.0,3.0,BR,True,[William Windom],None


In [19]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

tags = pd.read_csv('/kaggle/input/datasets/zvozvozvo/movielenseda/tags.csv')

print(f"Исходных тегов: {len(tags)}")

# Группируем теги по фильмам, объединяя их в одну строку
movie_descriptions = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(str(i) for i in x)).reset_index()

Исходных тегов: 2000072


In [20]:
# Кодирование (BERT)
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(movie_descriptions['tag'].tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1604 [00:00<?, ?it/s]

In [21]:
# Кластеризация
num_clusters = 30
print(f"Кластеризация на {num_clusters} групп")
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
movie_descriptions['cluster'] = kmeans.fit_predict(embeddings)

# Результат
result = movie_descriptions[['movieId', 'cluster']]

movies = movies.merge(result, on="movieId", how='left')
movies

Кластеризация на 30 групп


,movieId,title,genres,popularity,age,rating,runtimeMinutes,num_translations,main_region,is_multiregional,actors_list,director_name,cluster
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",0.001176,30,3.778556,81.0,66.0,IN,True,"[Tom Hanks, Tim Allen, Wallace Shawn, John Mor...",None,23.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",0.000645,30,3.143925,104.0,54.0,CA,True,"[Robin Williams, Kirsten Dunst, David Alan Gri...",None,23.0
2,3,Grumpier Old Men,"[Comedy, Romance]",0.000120,30,3.070352,101.0,35.0,JP,True,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",None,24.0
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",0.000027,30,2.803635,124.0,20.0,US,True,"[Angela Bassett, Loretta Devine, Dennis Haysbe...",Forest Whitaker,1.0
4,5,Father of the Bride Part II,[Comedy],0.000155,30,2.984719,106.0,34.0,US,True,"[Steve Martin, Diane Keaton, Martin Short, Jan...",None,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
84427,292731,The Monroy Affaire,[Drama],0.000000,3,4.000000,95.0,3.0,MX,True,[Damián Alcázar],None,NaN
84428,292737,Shelter in Solitude,"[Comedy, Drama]",0.000000,2,1.000000,93.0,7.0,AU,True,[Robert Patrick],None,NaN
84429,292753,Orca,[Drama],0.000000,2,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84430,292755,The Angry Breed,[Drama],0.000000,57,1.000000,94.0,3.0,BR,True,[William Windom],None,NaN


In [39]:
movies.to_csv('prepared_movies.csv', index=False)

In [37]:
# Переходим к этапу кандидатогенерации

# Чтобы избежать oom приедстя ограничить список фильмов
popular_movies = [i for i,j in sorted(most_pop.items(), key=lambda x: x[1], reverse=True)[:25000]]

train_df_pop = train_df[train_df["movieId"].isin(popular_movies)].copy()

In [41]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder

user_enc = LabelEncoder()
item_enc = LabelEncoder()

train_df_pop["user_id_x"] = user_enc.fit_transform(train_df_pop["userId"])
train_df_pop["item_id_x"] = item_enc.fit_transform(train_df_pop["movieId"])

num_users = train_df_pop["user_id_x"].nunique()
num_items = train_df_pop["item_id_x"].nunique()

train_df_pop.sample(5)

,userId,movieId,rating,timestamp,user_id_x,item_id_x
2984676,18806,852,3,1086045694,765,744
5621257,35065,6502,4,1121826820,1425,5981
14437793,90347,410,3,1170547027,3861,383
26006644,163381,100498,3,1425206378,6966,16261
8598718,53915,32,5,1442751581,2262,31


In [42]:
import scipy.sparse as sps

matrix = sps.coo_matrix(
    (np.ones(train_df_pop.shape[0]),
     (train_df_pop['user_id_x'], train_df_pop['item_id_x'])),
    shape=(num_users, num_items)
)

matrix

<COOrdinate sparse matrix of dtype 'float64'
	with 3641327 stored elements and shape (8567, 25000)>

In [43]:
def fit_ease(X, reg_weight=100):
    
    G = X.T @ X

    # L2-регуляризация: λI
    G += reg_weight * sps.identity(G.shape[0])

    # Преобразуем в dense
    G = G.todense()

    # Обратная матрица
    P = np.linalg.inv(G)

    # Формула EASE
    B = P / (-np.diag(P))

    # Нулевая диагональ
    np.fill_diagonal(B, 0.)

    return B

w = fit_ease(matrix)

In [44]:
w.shape

(25000, 25000)

In [51]:
joined

,userId,train_interactions,test_interactions
0,28,"[(8666, 0), (54, 1), (75, 1), (160, 1), (174, ...","[(4125, 0), (4138, 0), (261759, 1), (5679, 2),..."
1,35,"[(1247, 2), (1590, 2), (1917, 2), (6365, 2), (...","[(2513, 2), (5218, 2), (5618, 2), (7115, 2), (..."
2,100,"[(49274, 1), (60069, 1), (34, 2), (1127, 2), (...","[(8533, 1), (222477, 1), (608, 2), (1183, 2), ..."
3,111,"[(104, 2), (288, 2), (356, 2), (2424, 2), (316...","[(168330, 3), (158238, 4)]"
4,140,"[(94959, 2), (168254, 2), (6870, 3), (56587, 3...","[(160954, 3), (176101, 5), (177593, 5), (17776..."
...,...,...,...
8562,200898,"[(4306, 3), (115713, 3), (122904, 3), (152081,...","[(2662, 2), (6534, 3), (78469, 3), (85414, 3),..."
8563,200915,"[(88267, 0), (18, 1), (592, 1), (1676, 1), (21...","[(60069, 5)]"
8564,200920,"[(159721, 0), (56949, 1), (63992, 1), (72407, ...","[(179819, 3), (122906, 4), (122912, 5)]"
8565,200930,"[(2706, 0), (4718, 0), (122904, 0), (171729, 0...","[(112556, 0), (140880, 0), (176149, 0), (92259..."


In [ ]:
import numpy as np
import scipy.sparse as sps
from tqdm.notebook import tqdm

tqdm.pandas()

def get_ease_candidates(user_items_list, item_enc, w, top_k=100):
    # Извлекаем только movieId
    user_items = [t[0] for t in user_items_list]
    
    # Оставляем только те фильмы, которые есть в обученном LabelEncoder (item_enc)
    known_items = [t for t in user_items if t in item_enc.classes_]
    
    if len(known_items) == 0:
        return []

    # Кодируем movieId в индексы матрицы
    encoded = item_enc.transform(known_items)
    
    # Создаем вектор взаимодействий (строка 1 x N_items)
    n_items = w.shape[0]
    vector = np.zeros(n_items)
    vector[encoded] = 1
    
    # Вычисляем предсказания (1, N) * (N, N) -> (1, N)
    # Принудительно приводим к flat массиву для безопасности
    preds = np.asarray(vector @ w).flatten()

    # Маскируем просмотренные (чтобы не рекомендовать их снова)
    preds[encoded] = -1e9

    # Отбираем ТОП-K
    if top_k > len(preds):
        top_k = len(preds)
        
    top_indices = np.argsort(-preds)[:top_k]
    top_scores = preds[top_indices]
    
    # Декодируем индексы обратно в оригинальные movieId
    decoded_items = item_enc.inverse_transform(top_indices)
    
    # Возвращаем список кортежей (movieId, score)
    return list(zip(decoded_items, top_scores))

# Применяем
joined['candidates_with_scores'] = joined['train_interactions'].progress_apply(
    lambda x: get_ease_candidates(x, item_enc, w, top_k=100)
)

  0%|          | 0/8567 [00:00<?, ?it/s]

In [58]:
# Разворачиваем колонку со списками в отдельные строки
# Теперь вместо одного списка в ячейке будет один кортеж (item, score) в каждой строке
df_exploded = joined[['userId', 'candidates_with_scores']].explode('candidates_with_scores')

# Разбиваем кортеж (item, score) на две отдельные колонки
expanded_cols = pd.DataFrame(
    df_exploded['candidates_with_scores'].tolist(), 
    index=df_exploded.index,
    columns=['movieId', 'score']
)

# Объединяем userId из развернутого DF с новыми колонками
result = pd.concat([df_exploded['userId'], expanded_cols], axis=1)

result

,userId,movieId,score
0,28,70,0.199366
0,28,2424,0.190685
0,28,31878,0.183384
0,28,37729,0.169491
0,28,5014,0.167917
...,...,...,...
99,2742,141,0.172098
99,2742,1350,0.171622
99,2742,58103,0.171307
99,2742,1385,0.170890


In [61]:
result_df = result.merge(movies, on='movieId', how='left')
result_df


# result_df.to_csv('prepared_train_data.csv', index=False)

,userId,movieId,score,title,genres,popularity,age,rating,runtimeMinutes,num_translations,main_region,is_multiregional,actors_list,director_name,cluster
0,28,70,0.199366,From Dusk Till Dawn,"[Action, Comedy, Horror, Thriller]",0.000446,29,3.202508,108.0,51.0,CA,True,"[Harvey Keitel, George Clooney, Juliette Lewis...",Robert Rodriguez,23.0
1,28,2424,0.190685,You've Got Mail,"[Comedy, Romance]",0.000398,27,3.053162,119.0,58.0,CA,True,"[Tom Hanks, Meg Ryan, Greg Kinnear, Parker Pos...",None,23.0
2,28,31878,0.183384,Kung Fu Hustle (Gong fu),"[Action, Comedy]",0.000261,21,3.561326,99.0,55.0,IN,True,[Stephen Chow],Stephen Chow,3.0
3,28,37729,0.169491,Corpse Bride,"[Animation, Comedy, Fantasy, Musical, Romance]",0.000436,20,3.334100,77.0,60.0,CA,True,"[Johnny Depp, Helena Bonham Carter, Emily Wats...",Tim Burton,23.0
4,28,5014,0.167917,I Am Sam,[Drama],0.000189,24,3.365230,132.0,40.0,ES,True,"[Sean Penn, Michelle Pfeiffer, Dakota Fanning,...",None,23.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2742,141,0.172098,"Birdcage, The",[Comedy],0.000267,29,3.440627,117.0,49.0,ES,True,"[Robin Williams, Nathan Lane, Gene Hackman, Di...",Mike Nichols,23.0
9996,2742,1350,0.171622,"Omen, The","[Horror, Mystery, Thriller]",0.000187,49,3.445260,111.0,50.0,CA,True,"[Gregory Peck, David Warner, Billie Whitelaw]",Richard Donner,23.0
9997,2742,58103,0.171307,Vantage Point,"[Action, Drama, Thriller]",0.000134,17,3.034606,90.0,37.0,ES,True,"[Dennis Quaid, Forest Whitaker, Bruce McGill, ...",None,3.0
9998,2742,1385,0.170890,Under Siege,"[Action, Drama, Thriller]",0.000167,33,3.027050,103.0,51.0,FI,True,"[Steven Seagal, Gary Busey, Tommy Lee Jones, G...",Andrew Davis,23.0
